# C — Comumnicate

> Este notebook trata do que encontramos,o que fizemos e por quê importa.

---

**Afinal, o que o modelo faz? Funciona?**

## 1. Entendendo o problema

Em um banco, imagine que todo dia chegam 284 mil transações no sistema e, dessas, 492 são fraudes. Isso é 0,17%, menos de dois por mil e se o sistema bloquear toda transação acima de R$ 500 por precaução, vai pegar muita fraude, mas também vai bloquear compras inocentes, o que chamamos de falso positivo. Por outro lado, se o sistema for frouxo demais, as fraudes passam, o que é o falso negativo. Por isso, o objetivo é encontrar o equilíbrio certo.

> Trabalhamos com um dataset real de transações europeias de setembro de 2013, disponibilizado pela Worldline e pela Universidade Livre de Bruxelas (ULB) no Kaggle.
> Por razões de privacidade, os dados foram "embaralhados matematicamente" (via PCA) antes de serem publicados, então não sabemos o nome dos portadores nem os estabelecimentos. O que temos são 28 variáveis numéricas anônimas (V1 a V28), mais o valor da transação e o horário.

## 2. O que encontramos nos dados (antes de qualquer modelo)

### Fraudes preferem valores pequenos

Intuitivo, mas confirmado pelos dados: a mediana das transações fraudulentas é de apenas ~€9 contra ~€22 nas legítimas. Isso bate com uma prática comum chamada de quando alguém rouba um cartão e primeiro faz uma compra pequena para ver se o cartão está ativo antes de partir para compras maiores.
Mas também vimos fraudes que aparecem em valores altos, então o valor sozinho não é um bom detector.

### Horário importa, mas não do jeito óbvio

Fraudes acontecem em todos os horários, mas com um padrão da madrugada ter proporcionalmente mais fraude. O que faz sentido, porque é quando os donos dos cartões dormem e não recebem as notificações.

### Algumas variáveis anônimas separam muito bem as classes

As variáveis V14, V17 e V12 têm distribuições completamente diferentes entre fraudes e transações legítimas. Mesmo sem saber o que elas representam, o modelo consegue usar esse sinal para detectar fraudes com alta precisão.

## 3. Abordagem

Se você treinar um modelo sem muita atenção, ele vai aprender a dizer "legítima" para tudo (é, nesse caso ele vai estar certo em 99,83% das vezes, tendo em vista os poucos dados irregulares), mas ele nunca detecta fraude. 
Em vez de fingir que as classes são iguais (o que seria mentira), ensinamos o modelo que errar numa fraude é muito mais caro do que errar numa transação legítima. Matematicamente, a gente define que cada fraude vale quase 600 vezes mais do que uma transação comum durante o treinamento, então o modelo aprende, desde o início, que deixar passar uma fraude é grave.

---
> Por que não SMOTE? Essa técnica cria fraudes sintéticas para equilibrar as classes, mas o Cost-Sensitive tem uma vantagem enorme de não inventar dados ( e as fraudes sintéticas poderiam capturar padrões que não existem no mundo real), por isso preferimos trabalhar com o que realmente aconteceu.

## 4. Sobre modelos e acurácia

Testamos três abordagens:

| Modelo | O que é | PR-AUC |
|--------|---------|--------|
| Regressão Logística | Modelo linear clássico | 0.694 |
| Random Forest | Conjunto de árvores de decisão | 0.805 |
| XGBoost | Árvores que aprendem dos próprios erros | 0.810 |

O XGBoost ganhou, mas a margem sobre o Random Forest é pequena.

### Por que ignoramos a acurácia?

A acurácia mede quantas respostas estão certas no total, mas, como já dito, com 0,17% de fraudes qualquer modelo que diga "legítima" para tudo acerta em quase 100% e não detecta nada.
Em contrapartida, o PR-AUC (Área sob a Curva Precisão-Recall) mede especificamente quanto o modelo acerta nas fraudes, em diferentes limiares de decisão. É a métrica certa para problemas onde um erro de tipo importa muito mais do que o outro.

### Optuna

Usamos o Optuna (uma ferramenta de otimização automática) para encontrar a melhor configuração do XGBoost, mas a melhora foi de apenas +0.0004 em PR-AUC, dentro do ruído estatístico.
Contudo, isso não é um fracasso, certo? É um resultado nos dizendo que o dataset já tem tanto sinal que um modelo simples captura quase tudo. Em outros datasets, a otimização poderia fazer diferença maior.

## 5. Calibração

Aqui está algo que foi novidade pra nós e acho que não é muito comentado: um modelo de ML pode ser ótimo para rankear fraudes sem ser honesto sobre as probabilidades que ele retorna. Com o peso de 600× que colocamos nas fraudes durante o treino, o modelo aprendeu a ser muito agressivo e, como resultado, mais de metade das fraudes receberam score de exatamente 1.000, que é o máximo possível.
Se uma transação com 70% de chance de fraude e uma com 99% de chance chegar para o analista com o mesmo score de 1.000, por exemplo, ele não conseguiria priorizar.

---

Por isso, aplicamos uma técnica chamada Platt Scaling, na qual ajustamos uma regressão logística simples sobre os scores do modelo para que eles reflitam probabilidades reais.

Eis os resultados com limiar de decisão em 0.5:

| | Sem calibração | Com Platt Scaling |
|-|---------------|-------------------|
| Alertas gerados por dia | 84 | 74 |
| Desses alertas, % que são fraude real | 90,5% | 97,3% |
| Score máximo numa fraude | 1.000 (teto) | 0.918 (distribuído) |
| Poder de detectar fraudes (PR-AUC) | 0.820 | 0.815 |

Aquele mesmo analista recebe 10 alertas a menos por dia e cada um que ele investiga tem 97% de chance de ser fraude real. Além disso, a perda de cobertura é mínima (−0.005 em PR-AUC). Isso importa muito dado o tempo limitado de investigar essas coisas e um sistema que gera muitos falsos alarmes cansa.

## 6. Resumo

Se fôssemos recomendar uma pipeline para um banco, seria essa:

```
Transação nova
     ↓
Extrair Hour do timestamp (padrão circadiano)
     ↓
Normalizar Amount com RobustScaler (ajustado no histórico, não nos dados futuros)
     ↓
XGBoost com scale_pos_weight ≈ 600
     ↓
Platt Scaling (calibração de probabilidade)
     ↓
Score entre 0 e 1 (probabilidade real de fraude)
     ↓
Se score ≥ 0.5 → alerta para analista
Se score ≥ 0.9 → bloqueio automático (alta confiança)
```

### Números esperados (com base no teste)

- 74 alertas para ~57.000 transações (~1 em cada 770)
- 97% de precisão nos alertas gerados
- 79% das fraudes detectadas (recall), ou seja, de cada 10 fraudes que acontecem, o sistema pega quase 8

## Este projeto foi desenvolvido como trabalho acadêmico de Pensamento Analítico de Dados (PAD) na UFG, inspirado na metodologia AGEMC